In [1]:
# Automatically display how long each notebook cell takes to execute.
%load_ext autotime


time: 124 μs (started: 2026-10-03 09:19:11 +05:30)


In [2]:
import pickle
import time
from pathlib import Path

import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
import sklearn
from mlflow import MlflowClient
from mlflow.models import infer_signature
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    PrecisionRecallDisplay,
    RocCurveDisplay,
    average_precision_score,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

print(f"mlflow {mlflow.__version__} · scikit-learn {sklearn.__version__} · pandas {pd.__version__}")

c:\Users\Sharath\.conda\envs\agents\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


mlflow 3.16.1 · scikit-learn 1.9.1 · pandas 3.0.6
time: 3.15 s (started: 2026-10-03 09:19:12 +05:30)


In [ ]:
pd.options.display.max_columns = 100 # Show up to 100 DataFrame columns.
pd.options.display.max_colwidth = None # Don't truncate long content inside an individual cell.

time: 971 μs (started: 2026-10-03 09:19:26 +05:30)


In [4]:
bookings = pd.read_csv("data/bookings_2015_2017.csv.gz")

print(f"{len(bookings):,} bookings · arrivals {bookings.arrival_date.min()} → {bookings.arrival_date.max()}")
print(f"cancelled: {bookings.is_canceled.mean():.1%}")
bookings.head()

118,300 bookings · arrivals 2015-07-01 → 2017-08-24
cancelled: 37.1%


,booking_id,arrival_date,hotel,lead_time,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,meal,market_segment,distribution_channel,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,reserved_room_type,deposit_type,customer_type,days_in_waiting_list,adr,required_car_parking_spaces,total_of_special_requests,is_canceled
0,BK100001,2015-07-01,City Hotel,6,0,2,1,0.0,0,HB,Offline TA/TO,TA/TO,0,0,0,A,No Deposit,Transient,0,0.0,0,0,0
1,BK100002,2015-07-01,City Hotel,88,0,4,2,0.0,0,BB,Online TA,TA/TO,0,0,0,A,No Deposit,Transient,0,76.5,0,1,1
2,BK100003,2015-07-01,City Hotel,65,0,4,1,0.0,0,BB,Online TA,TA/TO,0,0,0,A,No Deposit,Transient,0,68.0,0,1,1
3,BK100004,2015-07-01,City Hotel,92,2,4,2,0.0,0,BB,Online TA,TA/TO,0,0,0,A,No Deposit,Transient,0,76.5,0,2,1
4,BK100005,2015-07-01,City Hotel,257,0,2,2,0.0,0,HB,Offline TA/TO,TA/TO,0,1,0,A,No Deposit,Transient,0,101.5,0,0,1


time: 356 ms (started: 2026-10-03 09:19:33 +05:30)


In [7]:
NUMERIC = [
    "lead_time", "stays_in_weekend_nights", "stays_in_week_nights", "adults", "children",
    "babies", "is_repeated_guest", "previous_cancellations", "previous_bookings_not_canceled",
    "days_in_waiting_list", "adr", "required_car_parking_spaces", "total_of_special_requests",
]
CATEGORICAL = [
    "hotel", "meal", "market_segment", "distribution_channel", "reserved_room_type",
    "deposit_type", "customer_type",
]
FEATURES = NUMERIC + CATEGORICAL
SEED = 42

X, y = bookings[FEATURES], bookings["is_canceled"]
X_train, X_rest, y_train, y_rest = train_test_split(X, y, test_size=0.4, stratify=y, random_state=SEED)
X_val, X_test, y_val, y_test = train_test_split(X_rest, y_rest, test_size=0.5, stratify=y_rest, random_state=SEED)

print(f"train {len(X_train):,} · validation {len(X_val):,} · test {len(X_test):,} · {len(FEATURES)} features")

train 70,980 · validation 23,660 · test 23,660 · 20 features
time: 119 ms (started: 2026-10-03 09:22:27 +05:30)


In [ ]:
def build_pipeline(params):
    # params = {"model_type": ..., **hyperparameters}  →  an untrained preprocessing + model Pipeline
    hyper = {k: v for k, v in params.items() if k != "model_type"}
    if params["model_type"] == "logreg":
        model = LogisticRegression(max_iter=1000, **hyper)
    elif params["model_type"] == "random_forest":
        # n_jobs=-1 :This tells scikit-learn to use all available CPU cores for operations that support parallelism.
        # random_state=SEED : This makes randomized behavior reproducible. helps you get consistent results across repeated runs, assuming the rest of the environment is controlled.
        model = RandomForestClassifier(n_jobs=-1, random_state=SEED, **hyper) # emember, ** unpacks a dictionary into keyword arguments.
    elif params["model_type"] == "hist_gb": 
        model = HistGradientBoostingClassifier(random_state=SEED, **hyper)
    # ColumnTransformer lets you apply different preprocessing to different columns.
    prep = ColumnTransformer([
        ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), NUMERIC),
        ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL),
    ])
    return Pipeline([("prep", prep), ("model", model)])

# handle_unknown="ignore"
# This is important for inference.
# Imagine training data contains:
# Urban
# Rural
# Semiurban

# but later new data contains a category the encoder didn't see during training.
# With:
# handle_unknown="ignore"
# the encoder won't crash just because an unseen category appears; it represents that category with zeros across the known one-hot columns for that feature.


time: 1.79 ms (started: 2026-10-03 09:22:35 +05:30)


In [ ]:
# This evaluate() function is a reusable binary-classification evaluation function. 
# It takes a trained pipeline, makes predictions, calculates five metrics, 
# and returns them as a dictionary.

# There are four inputs:

# pipe ->	Your trained sklearn Pipeline
# X ->	Features to evaluate
# y ->	Actual target labels
# prefix -> Prefix added to metric names; default is "val"

#get probability predictions
# proba = pipe.predict_proba(X)[:, 1]

# This is an important line.
# For binary classification, suppose your classes are:
# 0 → Loan Rejected
# 1 → Loan Approved

# predict_proba() returns probabilities for both classes.
# For example:
# pipe.predict_proba(X)

# could produce:
#                Class 0    Class 1
# Customer 1       0.20       0.80
# Customer 2       0.75       0.25
# Customer 3       0.40       0.60
# Customer 4       0.90       0.10

# In NumPy form:
# [    [0.20, 0.80],    [0.75, 0.25],    [0.40, 0.60],    [0.90, 0.10]]

# But you specifically want the probability of the positive class (1).
# That's what this does:
# [:, 1]

# Breaking it down:
# :   → select ALL rows
# 1   → select column at index 1

# Therefore:
# proba = pipe.predict_proba(X)[:, 1]

# becomes:
# proba = [    0.80,    0.25,    0.60,    0.10]

# These are the probabilities of class 1.

#############################################################
    
# What does .astype(int) do?
# Before conversion:
# proba >= 0.5


# produces booleans:
# [True, False, True, False]


# Then:
# .astype(int)


# converts:
# True  → 1
# False → 0

# giving:
# [1, 0, 1, 0]

##############################################################

def evaluate(pipe, X, y, prefix="val"):
    proba = pipe.predict_proba(X)[:, 1]
    pred = (proba >= 0.5).astype(int)
    return {
        f"{prefix}_roc_auc": roc_auc_score(y, proba),
        f"{prefix}_pr_auc": average_precision_score(y, proba),
        f"{prefix}_f1": f1_score(y, pred),
        f"{prefix}_precision": precision_score(y, pred),
        f"{prefix}_recall": recall_score(y, pred),
    }

time: 1.23 ms (started: 2026-10-03 10:33:37 +05:30)


In [ ]:
# 10. What does prefix do?
# This is a nice design in your function:
# prefix="val"


# You're dynamically creating metric names using Python f-strings.
# For example:
# f"{prefix}_f1"


# If:
# prefix = "val"


# it becomes:
# val_f1

# Therefore:
# evaluate(pipe, X_val, y_val)


# might return:
# {    "val_roc_auc": 0.91,    "val_pr_auc": 0.88,    "val_f1": 0.84,    "val_precision": 0.87,    "val_recall": 0.82}


# But now you can reuse the same function for training data:
# evaluate(    pipe,    X_train,    y_train,    prefix="train")


# which returns:
# {    "train_roc_auc": 0.96,    "train_pr_auc": 0.94,    "train_f1": 0.91,    "train_precision": 0.92,    "train_recall": 0.90}


# That's much better than writing two separate evaluation functions.
# This becomes very useful with MLflow

# You could do:
# train_metrics = evaluate(    pipe,    X_train,    y_train,    prefix="train")
# val_metrics = evaluate(    pipe,    X_val,    y_val,    prefix="val")


# Then:
# print(train_metrics)print(val_metrics)


# And because both are dictionaries, MLflow can conveniently log the metrics:
# mlflow.log_metrics(train_metrics)
# mlflow.log_metrics(val_metrics)


# Instead of writing:
# mlflow.log_metric("val_roc_auc", ...)
# mlflow.log_metric("val_pr_auc", ...)
# mlflow.log_metric("val_f1", ...)
# mlflow.log_metric("val_precision", ...)
# mlflow.log_metric("val_recall", ...)


# one by one.
# So the overall workflow becomes:
#              build_pipeline(params)
#                       ↓
#                    Pipeline
#                       ↓
#             pipe.fit(X_train,y_train)
#                       ↓
#                Trained Pipeline
#                       │
#             ┌─────────┴─────────┐
#             ↓                   ↓
#         Training             Validation
#             ↓                   ↓
#        evaluate()            evaluate()
#             ↓                   ↓
#      train_metrics          val_metrics
#             │                   │
#             └─────────┬─────────┘
#                       ↓
#                MLflow Metrics

In [ ]:
for C in [0.1, 1.0]:
    pipe = build_pipeline({"model_type": "logreg", "C": C}).fit(X_train, y_train)
    print(f"logreg C={C}:  val ROC-AUC {evaluate(pipe, X_val, y_val)['val_roc_auc']:.4f}")
    
# This part:
# :.4f

# is Python number formatting.
# It means: Display the floating-point number with 4 digits after the decimal point.

# For example:
# 0.87634572 -> 0.8763

# So:
# f"{0.87634572:.4f}"
# produces:
# 0.8763

logreg C=0.1:  val ROC-AUC 0.8471
logreg C=1.0:  val ROC-AUC 0.8474
time: 1.16 s (started: 2026-10-03 10:35:15 +05:30)


In [12]:
mlflow.set_tracking_uri("sqlite:///mlflow.db")

time: 867 μs (started: 2026-10-03 10:53:20 +05:30)


In [13]:
experiment = mlflow.set_experiment("booking-cancellations")

print("tracking URI     :", mlflow.get_tracking_uri())
print("experiment       :", experiment.name, "| id", experiment.experiment_id)
print("artifact location:", experiment.artifact_location)

2026/10/03 10:53:22 INFO mlflow.tracking.fluent: Experiment with name 'booking-cancellations' does not exist. Creating a new experiment.


tracking URI     : sqlite:///mlflow.db
experiment       : booking-cancellations | id 2
artifact location: file:d:/MLOPS_Shivam_prasad/MLOPS_Github_Code/v1-aiml-mlops-sep-2026/lec5 Experiment Tracking with MLFlow 1/mlruns/2
time: 1.04 s (started: 2026-10-03 10:53:21 +05:30)


In [14]:
params = {"model_type": "logreg", "C": 1.0}

with mlflow.start_run(
    run_name="baseline-logreg",
    description="Logistic regression on the 20 raw features: the bar every other model has to clear.",
) as run:
    mlflow.log_params(params)
    mlflow.set_tags({"stage": "baseline", "team": "revenue-analytics"})

    start = time.time()
    pipe = build_pipeline(params).fit(X_train, y_train)
    mlflow.log_metric("fit_time_s", time.time() - start)

    metrics = evaluate(pipe, X_val, y_val)
    mlflow.log_metrics(metrics)

print("run_id:", run.info.run_id)
pd.Series(metrics).round(4)

run_id: ce0d152ec0e74441ba33f35f618a750b


val_roc_auc      0.8474
val_pr_auc       0.8226
val_f1           0.6965
val_precision    0.8449
val_recall       0.5924
dtype: float64

time: 975 ms (started: 2026-10-03 10:55:40 +05:30)


In [ ]:
# This code retrieves an already logged MLflow run and prints its details: status, parameters, metrics, and tags.

# 2. logged.info vs logged.data
# The returned run has two particularly important sections:
# logged
# │
# ├── logged.info
# │      ↓
# │   Information ABOUT the run
# │
# └── logged.data
#        ↓
#     Information LOGGED INSIDE the run

# Think of them as:
# Object	Contains
# logged.info	Run ID, status, experiment ID, start/end time, etc.
# logged.data	Parameters, metrics and tags
logged = mlflow.get_run(run.info.run_id)

print("status :", logged.info.status)
print("params :", logged.data.params)
print("metrics:", {k: round(v, 4) for k, v in logged.data.metrics.items()}) # For every metric, keep its name and round its value to 4 decimal places.
print("tags   :")
for key, value in sorted(logged.data.tags.items()):
    print(f"   {key:<22} {value}") # Left-align key within a field that is 22 characters wide.

status : FINISHED
params : {'model_type': 'logreg', 'C': '1.0'}
metrics: {'fit_time_s': 0.5384, 'val_roc_auc': 0.8474, 'val_pr_auc': 0.8226, 'val_f1': 0.6965, 'val_precision': 0.8449, 'val_recall': 0.5924}
tags   :
   mlflow.note.content    Logistic regression on the 20 raw features: the bar every other model has to clear.
   mlflow.runName         baseline-logreg
   mlflow.source.name     Live_Class_Notebook.ipynb
   mlflow.source.type     NOTEBOOK
   mlflow.user            Sharath
   stage                  baseline
   team                   revenue-analytics
time: 13.3 ms (started: 2026-10-03 11:37:03 +05:30)


## Let's Sweep

In [ ]:
candidates = {
    "logreg-C0.1":       {"model_type": "logreg", "C": 0.1},
    "rf-200-d12":        {"model_type": "random_forest", "n_estimators": 200, "max_depth": 12},
    "rf-200-d20":        {"model_type": "random_forest", "n_estimators": 200, "max_depth": 20},
    "rf-200-d30":        {"model_type": "random_forest", "n_estimators": 200, "max_depth": 30},
    "rf-400-d30":        {"model_type": "random_forest", "n_estimators": 400, "max_depth": 30},
    "hist_gb-31leaves":  {"model_type": "hist_gb", "learning_rate": 0.1, "max_leaf_nodes": 31, "max_iter": 300},
    "hist_gb-127leaves": {"model_type": "hist_gb", "learning_rate": 0.1, "max_leaf_nodes": 127, "max_iter": 300},
}

for run_name, params in candidates.items():
    with mlflow.start_run(run_name=run_name) as run:
        mlflow.log_params(params) # stores all the parameters and values are converted to string
        mlflow.log_dict(params, "config.json") # it preserve all parameters in original data types
        mlflow.set_tag("stage", "sweep")

        start = time.time()
        pipe = build_pipeline(params).fit(X_train, y_train)
        fit_time = time.time() - start
        size_mb = len(pickle.dumps(pipe)) / 1e6
        metrics = evaluate(pipe, X_val, y_val)
        mlflow.log_metrics({**metrics, "fit_time_s": fit_time, "model_size_mb": size_mb})

        if params["model_type"] == "hist_gb":  # one loss value per boosting round → a curve
            booster = pipe.named_steps["model"] # You can access individual steps using: pipe.named_steps
            for step, (train_score, holdout_score) in enumerate(
                zip(booster.train_score_, booster.validation_score_)
            ):
                mlflow.log_metrics({"train_loss": -train_score, "holdout_loss": -holdout_score}, step=step)

    print(f"{run_name:<18} val_roc_auc {metrics['val_roc_auc']:.4f}   fit {fit_time:5.1f}s   {size_mb:7.1f} MB")

logreg-C0.1        val_roc_auc 0.8471   fit   0.5s       0.0 MB
rf-200-d12         val_roc_auc 0.8889   fit   2.3s      15.8 MB
rf-200-d20         val_roc_auc 0.9099   fit   2.4s     109.2 MB
rf-200-d30         val_roc_auc 0.9160   fit   2.7s     309.2 MB
rf-400-d30         val_roc_auc 0.9162   fit   5.0s     616.7 MB
hist_gb-31leaves   val_roc_auc 0.9040   fit   2.1s       1.1 MB
hist_gb-127leaves  val_roc_auc 0.9100   fit   3.4s       3.0 MB
time: 28.8 s (started: 2026-10-03 12:31:58 +05:30)


In [18]:
sweep = mlflow.search_runs(
    experiment_names=["booking-cancellations"],
    filter_string="tags.stage = 'sweep' and attributes.status = 'FINISHED'",
    order_by=["metrics.val_roc_auc DESC"],
)

sweep[["tags.mlflow.runName", "metrics.val_roc_auc", "metrics.val_pr_auc", "metrics.val_f1",
       "metrics.model_size_mb", "metrics.fit_time_s"]].round(3)

,tags.mlflow.runName,metrics.val_roc_auc,metrics.val_pr_auc,metrics.val_f1,metrics.model_size_mb,metrics.fit_time_s
0,rf-400-d30,0.916,0.894,0.785,616.669,4.968
1,rf-200-d30,0.916,0.894,0.784,309.166,2.729
2,hist_gb-127leaves,0.910,0.885,0.768,2.961,3.359
3,rf-200-d20,0.910,0.884,0.761,109.153,2.395
4,hist_gb-31leaves,0.904,0.877,0.756,1.079,2.120
5,rf-200-d12,0.889,0.859,0.714,15.815,2.298
6,logreg-C0.1,0.847,0.822,0.696,0.005,0.499


time: 62.9 ms (started: 2026-10-03 12:40:33 +05:30)
